# GPU programming is cost-model first
### A one-hour tour for computer architects meeting the GPU

> **You don't pick an algorithm and make the chip run it. You compute the cost model first** --
> bytes moved, passes over memory, latency vs concurrency, where data lives, how you touch it --
> **and *that* selects the algorithm before you write a line.** Asymptotic optimality is not hardware-neutral.

The chip: a single **RTX 6000 Ada** (sm_89). Every claim below is **measured live** on it.

In [1]:
# setup: live-CUDA cells (%%cuda) + plotting. nvcc/ncu on PATH; arch set once.
import os, subprocess, re
import matplotlib.pyplot as plt
ROOT = os.getcwd()
get_ipython().run_line_magic("load_ext", "nvcc4jupyter")
from nvcc4jupyter import set_defaults
set_defaults(compiler_args="-arch=sm_89 -O3 -std=c++17")
def sh(cmd, cwd=None):
    return subprocess.run(cmd, shell=True, cwd=cwd or ROOT, capture_output=True, text=True).stdout
def grab(t, p):
    m = re.search(p, t); return float(m.group(1)) if m else float("nan")
print("ready: %%cuda live cells enabled.")

Source files will be saved in "/tmp/tmp5i_bzdg0".
ready: %%cuda live cells enabled.


### 0.1 The bet: throughput, not latency
A CPU core spends its area making *one* instruction stream fast (out-of-order, branch prediction, big caches).
A GPU rips that out and spends the area on **many ALUs** + **enough resident threads to switch among**.
Give up single-thread latency; buy aggregate throughput. Everything weird follows from this one bet.

![CPU spends area on control+cache for one fast thread; the GPU on a sea of ALUs + a huge register file.](slides/figures/01_area.png)

### 0.2 SIMT, and the words that mislead
32 lanes share one fetch/decode/scheduler (a **warp**) and run the same instruction in lockstep, each on its
*own* scalar registers. That is **SIMD execution with a per-lane register file** -- width is *across threads*,
not a wide register (unlike AVX). Pin the vocabulary once, or everything later is noise:

| you'll hear | it really is | CPU analogy |
|---|---|---|
| "CUDA core" (18,176!) | an **ALU / lane** | one SIMD lane |
| thread | one lane's scalar work | a lane of a vector op |
| warp | **32 lanes** sharing an instruction | one SIMD instruction |
| **SM** | the real processor (~142) | **the core** |
| register | per-lane; the file is partitioned by occupancy | register file |

One sentence: **SM = the core; warp = a SIMD instruction; "CUDA core" = a lane; thread = one lane's work.**

![AVX: one thread, one wide register. SIMT: 32 threads, one shared instruction, per-lane scalar registers.](slides/figures/02_simt_vs_avx.png)

### 0.3 Each op is slow -> hide it by oversubscription
No out-of-order engine, so a dependent global load is **hundreds of cycles** and the thread just stalls.
The GPU hides it by keeping **dozens of warps resident** and switching to a ready one the instant one stalls --
free, because every resident warp's registers live on-chip. (We prove this on *one* SM in §2.)
And memory bandwidth (~960 GB/s) only materializes if a warp's 32 lanes touch **contiguous** addresses
(**coalescing**, §4). Registers (~36 MB) and shared memory (programmer-managed scratchpad) are the fast tiers;
L2 (96 MB) and GDDR are the slow ones.

![A stalled warp is covered by another resident warp -- zero-overhead switch; the SM is never idle.](slides/figures/05_latency_hiding.png)

### 0.4 The programming model, in one kernel
Before the detail code, map every token to the hardware. A kernel is **scalar per-thread code**; the launch
`<<<grid, block>>>` stamps out `grid x block` threads; each finds its element by index; memory-space keywords
say **where a variable lives**. Run it:

In [2]:
%%cuda
#include <cstdio>
__global__ void saxpy(int n, float a, const float* x, float* y){
  int i = blockIdx.x * blockDim.x + threadIdx.x;   // (block, thread) -> a global element index
  if (i < n) y[i] = a*x[i] + y[i];                  // each lane: ordinary scalar code on ITS i
}                                                    // __global__ = runs on device, called from host
int main(){
  int n = 1<<20; size_t b = n*sizeof(float);
  float *x,*y; cudaMallocManaged(&x,b); cudaMallocManaged(&y,b);   // host<->device memory
  for(int i=0;i<n;i++){ x[i]=1.0f; y[i]=2.0f; }
  saxpy<<<(n+255)/256, 256>>>(n, 3.0f, x, y);       // grid of blocks x 256 threads/block (=warps of 32)
  cudaDeviceSynchronize();                          // host waits for the device
  printf("y[0]=%.1f (expect 5.0)  -- %d threads = %d blocks x 256\n", y[0], n, (n+255)/256);
}

y[0]=5.0 (expect 5.0)  -- 1048576 threads = 4096 blocks x 256



That's the whole model: **grid -> block -> warp(32) -> thread/lane**, `__global__`/`__device__`/`__shared__`/registers
for *where data lives*, and a host that launches + synchronizes. Everything past here is making this *fast*.

## 1. Bandwidth is the chip's identity  *(~4 min)*

> **Demo** `a streaming copy` &middot; **in:** one ~1 GB array &middot; **out:** achieved GB/s &middot; **why:** the simplest possible kernel, so the only thing measured is the bus.

The chip's native unit of work is **moving bytes**. A trivial `out[i]=in[i]` (vectorized to 128-bit `float4`)
saturates the GDDR6 bus; the number it hits is the yardstick every later demo is measured against.

**Guess first** 🎲 -- a GPU streaming copy vs a CPU STREAM-Triad -- how many times the bandwidth?

<table style="width:100%"><tr><td style="width:50%;vertical-align:top"><b>CPU STREAM Triad</b><pre>a[i] = b[i] + s*c[i]   // ~100s GB/s</pre></td><td style="width:50%;vertical-align:top"><b>GPU copy</b><pre>out[i] = in[i]        // ~?? GB/s</pre></td></tr></table>

In [3]:
%%cuda
#include <cstdio>
__global__ void copy_kernel(float4* __restrict__ out, const float4* __restrict__ in, size_t n4){
  size_t i = blockIdx.x*(size_t)blockDim.x + threadIdx.x, stride = (size_t)gridDim.x*blockDim.x;
  for (; i < n4; i += stride) out[i] = in[i];        // 128-bit coalesced loads/stores
}
int main(){
  size_t n = (size_t)1<<28, bytes = n*sizeof(float);  // 1 GB
  float *in,*out; cudaMalloc(&in,bytes); cudaMalloc(&out,bytes); cudaMemset(in,1,bytes);
  size_t n4 = n/4; int blk=256, grid=(int)((n4+blk-1)/blk); if(grid>65535) grid=65535;
  cudaEvent_t a,b; cudaEventCreate(&a); cudaEventCreate(&b);
  copy_kernel<<<grid,blk>>>((float4*)out,(const float4*)in,n4);   // warm up
  cudaEventRecord(a); copy_kernel<<<grid,blk>>>((float4*)out,(const float4*)in,n4); cudaEventRecord(b);
  cudaEventSynchronize(b); float ms=0; cudaEventElapsedTime(&ms,a,b);
  printf("GPU copy: %.0f GB/s  (read+write %zu MB in %.2f ms)\n", 2.0*bytes/1e9/(ms/1e3), 2*bytes/(1<<20), ms);
}

GPU copy: 815 GB/s  (read+write 2048 MB in 2.64 ms)



**Cost model.** 
A copy moves `2*N` bytes once; time = bytes / bandwidth. There is no algorithm here -- just the bus.
Everything later is "how close to this number can a *real* computation get?"